# Galerie GenAI — 01 · RAG fondations : construire ET évaluer 🟢

> ⚠️ **À faire APRÈS ton cas d'usage certif** — le sujet certif est du ML
> classique ; un RAG n'y a pas sa place (relis la grille C4). Cette galerie
> prépare tes **futurs** projets.
>
> **Étagère optionnelle** — pas un brief, pas de livrable, pas de note.
> **Autonomie** : 🟢 **résolu** — tu lis, tu exécutes, tu comprends chaque cellule.
> **Durée** : ~2 h 30 (dont ~5 min de téléchargements au premier lancement)
> **Fiches à garder ouvertes** : `panorama_genai_llm_rag_agents.md` ·
> `fiche_techniques_rag.pdf` · `grille_decision_C4.md`
> **Livre de référence** : *RAG made simple* (N. Diamant) — ce notebook
> correspond à son **chapitre 1** (Simple RAG) et son **chapitre 22**
> (RAG Evaluation). Repo compagnon : `github.com/NirDiamant/RAG_Techniques`.

## Le contexte

**FastIA** croule sous ses documents internes : politique télétravail, notes
de frais, procédures IT, comptes rendus projet… Les consultants posent sans
arrêt les mêmes questions aux RH et à la DSI.

> « On veut un assistant qui répond à partir de NOS documents, qui **cite sa
> source**, et qui **dit quand il ne sait pas**. Pas un chatbot qui invente. »

C'est un cas d'école du **RAG** (Retrieval-Augmented Generation) : plutôt que
d'espérer qu'un LLM connaisse tes documents (il ne les connaît pas), on
**cherche** les passages pertinents puis on demande au modèle de répondre
**uniquement à partir d'eux**.

Le geste central de ce notebook n'est pas de faire tourner un RAG — c'est de
l'**évaluer** : un RAG n'est pas fini quand il répond, il est fini quand tu
sais **mesurer** qu'il répond juste.

## Setup — choisis ton modèle selon ta machine

La génération passe par **Ollama** en local. Choisis selon ta machine (le
notebook fonctionne à l'identique, seule la qualité de rédaction change) :

| Ta machine | Modèle conseillé | Alternative | Commande |
|---|---|---|---|
| < 8 Go RAM | `llama3.2:1b` | `qwen2.5:0.5b` | `ollama pull llama3.2:1b` |
| 8-16 Go RAM (cas courant) | `qwen2.5:1.5b` | `llama3.2:3b` | `ollama pull qwen2.5:1.5b` |
| ≥ 16 Go / Apple Silicon récent | `qwen2.5:7b` | `mistral:7b` | `ollama pull qwen2.5:7b` |
| Ollama impossible sur ta machine | **MOCK_MODE** | — | rien à installer |

En **MOCK_MODE**, la génération est remplacée par un gabarit déterministe :
tu perds la rédaction, tu gardes 100 % du geste RAG (découpage, indexation,
recherche, **évaluation** — tout le reste est réel).

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import requests
import unicodedata

RANDOM_STATE = 42
OLLAMA_URL = "http://localhost:11434"
OLLAMA_MODEL = os.getenv("OLLAMA_MODEL", "qwen2.5:1.5b")
MOCK_MODE = os.getenv("MOCK_MODE", "0") == "1"

if not MOCK_MODE:
    try:
        modeles = [m["name"] for m in requests.get(f"{OLLAMA_URL}/api/tags",
                                                   timeout=3).json()["models"]]
        if OLLAMA_MODEL not in modeles:
            print(f"⚠️ {OLLAMA_MODEL} absent (installés : {modeles}) — "
                  f"fais `ollama pull {OLLAMA_MODEL}` ou change OLLAMA_MODEL.")
        else:
            print(f"✅ Ollama joignable, modèle : {OLLAMA_MODEL}")
    except Exception:
        MOCK_MODE = True
        print("⚠️ Ollama injoignable → bascule automatique en MOCK_MODE.")
else:
    print("MOCK_MODE actif : retrieval réel ; métriques de génération/citation non représentatives.")

## [1] Le corpus — 13 documents internes FastIA

Des documents **fictifs mais réalistes** (RH, IT, projets), fournis dans
`corpus_fastia/`. C'est petit exprès : tu peux tout lire, donc tu peux
**vérifier à la main** ce que le système fait — réflexe à garder sur un
vrai corpus de 10 000 documents… où tu ne pourras plus.

In [ ]:
DOSSIER_CORPUS = Path("corpus_fastia")

documents = {p.name: p.read_text(encoding="utf-8")
             for p in sorted(DOSSIER_CORPUS.glob("*.md"))}
print(f"{len(documents)} documents chargés")
for nom, texte in list(documents.items())[:5]:
    print(f"  {nom:42s} {len(texte):5d} caractères")

## [2] Découper : le chunking, premier choix structurant

Un document entier est trop long pour être « une unité de recherche ». On
découpe en **chunks**. Deux stratégies à comparer (le match sera arbitré par
l'évaluation en [6], pas par l'intuition) :

1. **Taille fixe** : 500 caractères, chevauchement de 100 — simple, aveugle
   à la structure : ça coupe au milieu des phrases et des tableaux ;
2. **Par sections** : on suit les titres `##` du markdown — chaque chunk est
   une unité de sens, et on lui préfixe son origine (*document > section*),
   ce qui l'ancre quand il sera lu isolément.

> 📖 *RAG made simple* : le découpage sémantique va plus loin (ch. 9), les
> en-têtes contextuels aussi (ch. 7) — ce sera le menu du notebook 02.

In [ ]:
def chunks_taille_fixe(nom_doc: str, texte: str, taille: int = 500,
                       chevauchement: int = 100) -> list[dict]:
    chunks, debut = [], 0
    while debut < len(texte):
        morceau = texte[debut:debut + taille]
        chunks.append({"texte": morceau, "source": nom_doc})
        debut += taille - chevauchement
    return chunks


def chunks_par_sections(nom_doc: str, texte: str) -> list[dict]:
    titre_doc = texte.splitlines()[0].lstrip("# ").strip()
    sections = texte.split("\n## ")
    chunks = []
    for section in sections[1:]:  # sections[0] = préambule avant le premier ##
        titre_section, _, corps = section.partition("\n")
        contenu = f"{titre_doc} > {titre_section.strip()}\n{corps.strip()}"
        chunks.append({"texte": contenu, "source": nom_doc})
    return chunks


corpus_fixe = [c for nom, txt in documents.items()
               for c in chunks_taille_fixe(nom, txt)]
corpus_sections = [c for nom, txt in documents.items()
                   for c in chunks_par_sections(nom, txt)]
print(f"taille fixe : {len(corpus_fixe)} chunks | par sections : {len(corpus_sections)} chunks")
print("\n--- exemple de chunk coupé n'importe où (taille fixe) ---")
print(corpus_fixe[1]["texte"][:180], "…")
print("\n--- exemple de chunk par section ---")
print(corpus_sections[0]["texte"][:180], "…")

## [3] Vectoriser : l'embedding, ou « le sens devient un vecteur »

Un modèle d'embedding transforme un texte en vecteur : deux textes proches
en **sens** donnent deux vecteurs proches en **cosinus** — même sans mot en
commun. On prend un modèle **multilingue léger** qui tourne sur CPU
(~470 Mo au premier téléchargement).

Avant d'indexer quoi que ce soit, démystifions sur 3 phrases :

In [ ]:
from sentence_transformers import SentenceTransformer

modele_embedding = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

phrases = [
    "Combien de jours de congés ai-je droit ?",
    "Quel est le nombre de jours de vacances par an ?",
    "Le certificat VPN expire au bout de douze mois.",
]
vecteurs = modele_embedding.encode(phrases, normalize_embeddings=True)
print("dimension d'un vecteur :", vecteurs.shape[1])
print("similarité congés/vacances  :", round(float(vecteurs[0] @ vecteurs[1]), 3))
print("similarité congés/VPN       :", round(float(vecteurs[0] @ vecteurs[2]), 3))

« Congés » et « vacances » se ressemblent pour le modèle **sans partager un
seul mot-clé** — c'est ça, la recherche sémantique, et c'est ce qu'un
`grep` ou un moteur par mots-clés ne sait pas faire.

## [4] Indexer et chercher : ChromaDB

Une base vectorielle stocke les couples (vecteur, texte, métadonnées) et
répond à « les k chunks les plus proches de cette question ». On indexe
**les deux corpus** pour pouvoir les comparer loyalement.

In [ ]:
import chromadb

client = chromadb.EphemeralClient()  # en mémoire : rien n'est écrit sur disque


def indexe(nom_collection: str, chunks: list[dict]):
    collection = client.create_collection(nom_collection,
                                          metadata={"hnsw:space": "cosine"})
    vecteurs = modele_embedding.encode([c["texte"] for c in chunks],
                                       normalize_embeddings=True,
                                       show_progress_bar=False)
    collection.add(ids=[f"{nom_collection}-{i}" for i in range(len(chunks))],
                   embeddings=vecteurs.tolist(),
                   documents=[c["texte"] for c in chunks],
                   metadatas=[{"source": c["source"]} for c in chunks])
    return collection


index_fixe = indexe("fixe", corpus_fixe)
index_sections = indexe("sections", corpus_sections)


def cherche(question: str, collection, k: int = 3) -> list[dict]:
    vecteur = modele_embedding.encode([question], normalize_embeddings=True)
    resultat = collection.query(query_embeddings=vecteur.tolist(), n_results=k)
    return [{"texte": d, "source": m["source"], "similarite": round(1 - dist, 3)}
            for d, m, dist in zip(resultat["documents"][0],
                                  resultat["metadatas"][0],
                                  resultat["distances"][0])]


for extrait in cherche("Quel est le plafond d'un repas du midi en déplacement ?",
                       index_sections):
    print(f"[{extrait['similarite']}] {extrait['source']:38s} {extrait['texte'][:70]}…")

## [5] Générer avec citations — le R, le A et le G réunis

Le prompt est le **contrat** passé avec le modèle, trois clauses non
négociables pour un assistant documentaire :

1. réponds **uniquement** à partir des extraits fournis ;
2. **cite ta source** au format `[source: nom_du_fichier]` ;
3. si les extraits ne suffisent pas, réponds exactement
   **« Hors de ma base documentaire. »** — l'aveu d'ignorance est une
   fonctionnalité, pas un échec.

In [ ]:
CONSIGNE_SYSTEME = (
    "Tu es l'assistant documentaire interne de FastIA. Réponds en français, "
    "en 1 à 3 phrases, UNIQUEMENT à partir des extraits fournis. Termine ta "
    "réponse par la citation [source: nom_du_fichier]. Si les extraits ne "
    "permettent pas de répondre, réponds exactement : "
    "\"Hors de ma base documentaire.\""
)


def genere(question: str, extraits: list[dict]) -> str:
    if MOCK_MODE:
        # gabarit déterministe : premier paragraphe du meilleur extrait + citation
        meilleur = extraits[0]
        resume = meilleur["texte"].split("\n")[0][:200]
        return f"D'après la documentation : {resume} [source: {meilleur['source']}]"
    bloc = "\n\n".join(f"--- extrait ({e['source']}) ---\n{e['texte']}"
                        for e in extraits)
    reponse = requests.post(f"{OLLAMA_URL}/api/chat", timeout=180, json={
        "model": OLLAMA_MODEL, "stream": False,
        "options": {"temperature": 0, "num_predict": 220},
        "messages": [
            {"role": "system", "content": CONSIGNE_SYSTEME},
            {"role": "user", "content": f"Extraits :\n{bloc}\n\nQuestion : {question}"},
        ],
    })
    reponse.raise_for_status()
    return reponse.json()["message"]["content"].strip()


question_demo = "Quelle est la durée de validité du certificat VPN ?"
print("Q :", question_demo)
print("R :", genere(question_demo, cherche(question_demo, index_sections)))

## [6] ÉVALUER — le geste qui distingue un POC d'un système

Sans évaluation, tu ne sauras jamais si un changement (autre chunking, autre
modèle, autre k) **améliore ou dégrade**. On fournit un jeu de **15 questions
dont on connaît le document-réponse** (`eval/questions_eval.json`) et on
mesure, exactement comme tu mesurais un f1_macro :

- **hit@1** : le bon document est-il le 1er résultat de la recherche ?
- **hit@3** : est-il dans le top 3 ? (c'est ce que voit le générateur)
- **citation correcte** : la réponse générée cite-t-elle le bon document ?

> 📖 C'est une version minimale du ch. 22 (*RAG Evaluation*) de
> *RAG made simple* — l'esprit est le même : un jeu de référence figé,
> des métriques par étage (retrieval / génération), comparées à chaque
> changement.

In [ ]:
import json

jeu_eval = json.loads(Path("eval/questions_eval.json").read_text(encoding="utf-8"))
questions_eval = jeu_eval["questions"]
print(f"{len(questions_eval)} questions d'évaluation — harnais didactique minimal, pas validation de production — exemple :")
print(" ", questions_eval[0]["question"], "→", questions_eval[0]["doc_attendu"])


def evalue_retrieval(collection, k: int = 3) -> pd.DataFrame:
    lignes = []
    for q in questions_eval:
        extraits = cherche(q["question"], collection, k=k)
        sources = [e["source"] for e in extraits]
        lignes.append({"question": q["question"][:45] + "…",
                       "doc_attendu": q["doc_attendu"],
                       "hit@1": sources[0] == q["doc_attendu"],
                       "hit@3": q["doc_attendu"] in sources})
    return pd.DataFrame(lignes)


detail_fixe = evalue_retrieval(index_fixe)
detail_sections = evalue_retrieval(index_sections)

bilan = pd.DataFrame({
    "chunking taille fixe": detail_fixe[["hit@1", "hit@3"]].mean().round(2),
    "chunking par sections": detail_sections[["hit@1", "hit@3"]].mean().round(2),
})
bilan

In [ ]:
# Où le retrieval se trompe-t-il encore ? (regarde les questions, pas que le score)
detail_sections[~detail_sections["hit@1"]]

Le chunking **par sections** domine le taille-fixe — non parce qu'il est
« plus malin », mais parce que chaque chunk est une unité de sens complète
avec son origine en en-tête. Retiens la **méthode** : c'est l'évaluation qui
a tranché, pas l'intuition. Au notebook 02, chaque technique du menu devra
battre ce score pour mériter sa place.

Maintenant, l'étage génération :

In [ ]:
def texte_normalise(texte: str) -> str:
    return " ".join(
        unicodedata.normalize("NFKD", texte).encode("ascii", "ignore").decode().lower().split()
    )


def evalue_generation(collection, k: int = 3) -> pd.DataFrame:
    lignes = []
    for q in questions_eval:
        extraits = cherche(q["question"], collection, k=k)
        reponse = genere(q["question"], extraits)
        lignes.append({"question": q["question"][:45] + "…",
                       "cite_le_bon_doc": q["doc_attendu"] in reponse,
                       "contient_le_fait_attendu": texte_normalise(q["reponse_attendue"]) in texte_normalise(reponse),
                       "reponse": reponse[:90] + "…"})
    return pd.DataFrame(lignes)


detail_gen = evalue_generation(index_sections)
print(f"Citations correctes : {detail_gen['cite_le_bon_doc'].mean():.0%}")
print(f"Réponses contenant le fait attendu : {detail_gen['contient_le_fait_attendu'].mean():.0%}")
detail_gen.head(8)

Deux lectures avant de t'inquiéter de ton score :

- **En MOCK_MODE**, la citation vient mécaniquement du meilleur extrait :
  ce taux ≈ ton hit@1 — normal ; ce n'est **pas** une mesure de génération ou d'ancrage.
- **Avec Ollama**, le taux dépend de la **taille du modèle** : un 1B suit
  moins bien la consigne de citation qu'un 7B (réponse juste mais citation
  absente ou mal formatée, voire réponse vide de temps en temps). C'est tout
  l'intérêt des **métriques par étage** : si hit@3 est bon mais la citation
  faible, le problème est dans la génération (consigne, modèle), pas dans la
  recherche — tu sais OÙ agir. Le contrôle du fait attendu est indicatif : une
  évaluation de production doit aussi vérifier l'ancrage de chaque affirmation
  (revue humaine ou juge calibré).

## [7] Le piège du silence — quand le RAG a l'air de savoir

Trois questions **dont la réponse n'existe pas dans le corpus** (prime de
cooptation, télétravail à l'étranger, congé sabbatique). Le danger : la
recherche vectorielle renvoie **toujours** ses k plus proches voisins — même
quand « le plus proche » est très loin. Et un LLM sans garde-fou rédigera
une réponse plausible par-dessus.

In [ ]:
questions_pieges = jeu_eval["hors_perimetre"]

similarites_normales = [cherche(q["question"], index_sections, k=1)[0]["similarite"]
                        for q in questions_eval]
print(f"Questions DANS le corpus  : similarité top-1 moyenne = "
      f"{np.mean(similarites_normales):.3f} (min {np.min(similarites_normales):.3f})")

for q in questions_pieges:
    extrait = cherche(q["question"], index_sections, k=1)[0]
    print(f"\nPiège : {q['question']}")
    print(f"  → meilleur candidat : {extrait['source']} (similarité {extrait['similarite']})")

In [ ]:
# Sans garde-fou : on force la génération sur le meilleur candidat trouvé.
piege = questions_pieges[1]["question"]  # télétravail depuis l'étranger
extraits_piege = cherche(piege, index_sections)
print("Q :", piege)
print("R (avec la consigne d'abstention) :", genere(piege, extraits_piege))

Regarde les chiffres : les pièges montent à ~0,5 de similarité, alors que la
question légitime la plus difficile descend à ~0,38. **Les deux distributions
se chevauchent** — et c'est le cas général en vrai projet, parce que les
questions hors périmètre portent sur les mêmes thèmes que le corpus
(cooptation ↔ RH, sabbatique ↔ congés).

Deux garde-fous complémentaires, à toujours poser ensemble :

1. **La consigne d'abstention** dans le prompt (déjà en place) — nécessaire
   mais fragile : elle repose sur la discipline du modèle ;
2. **Un seuil de similarité** AVANT la génération — mécanique, mais puisqu'il
   n'y a pas de séparation nette, c'est un **compromis à mesurer**, pas une
   valeur magique. Tu connais déjà ce raisonnement : c'est ton compromis
   précision ↔ rappel, version RAG.

In [ ]:
lignes = []
sims_pieges = [cherche(q["question"], index_sections, k=1)[0]["similarite"]
               for q in questions_pieges]
for seuil in [0.35, 0.40, 0.45, 0.50, 0.55]:
    lignes.append({
        "seuil": seuil,
        "pièges bloqués": f"{sum(s < seuil for s in sims_pieges)}/{len(sims_pieges)}",
        "questions légitimes rejetées à tort":
            f"{sum(s < seuil for s in similarites_normales)}/{len(similarites_normales)}",
    })
pd.DataFrame(lignes).set_index("seuil")

Il n'existe **aucun seuil qui bloque tout sans dommage collatéral**. Lis le
tableau comme un compromis précision ↔ rappel : ici on retient **0,45** —
il bloque le piège le plus net au prix d'une question légitime rejetée à
tort, et les pièges qui passent restent à la charge de la **consigne
d'abstention**. On choisit, on **documente**, et on complète : le notebook 02
ajoutera un **juge d'ancrage** (*Reliable RAG*, ch. 3 du livre) qui vérifie
la réponse APRÈS génération — la troisième ligne de défense.

In [ ]:
SEUIL_SIMILARITE = 0.45  # compromis choisi ET documenté grâce au tableau ci-dessus


def repond(question: str, collection, k: int = 3) -> str:
    extraits = cherche(question, collection, k=k)
    if extraits[0]["similarite"] < SEUIL_SIMILARITE:
        return ("Hors de ma base documentaire. "
                f"(similarité max {extraits[0]['similarite']} < seuil {SEUIL_SIMILARITE})")
    return genere(question, extraits)


for q in [questions_eval[0]["question"], questions_pieges[0]["question"]]:
    print("Q :", q)
    print("R :", repond(q, index_sections), "\n")

> ⚠️ Le seuil se **cale sur tes données** avec ton jeu d'éval (exactement
> comme le seuil de décision du notebook 02 de la galerie données mixtes se
> calait sur la validation) — il n'y a pas de valeur universelle. Et il se
> **surveille en prod** : si la part de « hors base » explose, c'est que les
> questions des utilisateurs ont dérivé par rapport au corpus — le drift,
> version RAG.

## 📝 Verdict — à rédiger

En 5 lignes, comme toujours : chunking retenu et pourquoi (chiffres à
l'appui), taux de citations correctes, comportement sur les questions
pièges, et LA limite que tu annoncerais au client avant la mise en service.

## 🔎 Ce que tu viens de pratiquer

- **L'anatomie complète d'un RAG** : chunking → embeddings → base
  vectorielle → retrieval → génération contrainte avec citations.
- **L'évaluation d'abord** : un jeu de questions-réponses de référence, des
  métriques par étage (hit@k pour la recherche, citation pour la
  génération) — AVANT de vouloir améliorer quoi que ce soit.
- **Le chunking est un choix mesurable**, pas un détail : sections > taille
  fixe ici, et c'est l'éval qui le dit.
- **Le piège du silence** : un RAG répond toujours quelque chose — abstention
  par consigne + seuil de similarité, les deux, toujours.

## ⭐ Pour aller plus loin (optionnel)

- Fais varier `k` (1, 3, 5, 8) et re-mesure : plus de contexte aide-t-il
  toujours la génération ?
- Remplace le modèle d'embedding par `intfloat/multilingual-e5-small` et
  compare le hit@3 (attention : e5 attend les préfixes `query:` /
  `passage:`).
- Ajoute 5 questions à `eval/questions_eval.json` sur des documents que le
  système rate — c'est comme ça qu'un jeu d'éval grandit en vrai projet.
- Passe au **notebook 02** : ton RAG rate encore des questions — le menu des
  techniques (query rewriting, reranking, fusion…) sert exactement à ça,
  symptôme par symptôme.